In [8]:
import glob
import os
import numpy as np
import rasterio
from rasterio.merge import merge
from rasterio.fill import fillnodata
from scipy import ndimage as ndi

paths = sorted(glob.glob("/Users/willmcmahon/Downloads/NL/*.tiff"))
print(f"Found {len(paths)} files to merge")
srcs = [rasterio.open(p) for p in paths]
mosaic, transform = merge(srcs)
profile = srcs[0].profile.copy()
profile.update(height=mosaic.shape[1], width=mosaic.shape[2], transform=transform)
nodata = srcs[0].nodata if srcs[0].nodata is not None else -9999
for s in srcs:
    s.close()

arr = mosaic[0]
missing = arr == nodata

# opening wipes out the small speckle and leaves only the big empty areas
border = ndi.binary_opening(missing, structure=np.ones((3, 3)), iterations=12)

filled = fillnodata(arr, mask=(~missing).astype("uint8"),
                    max_search_distance=50, smoothing_iterations=1)
filled[border] = nodata

profile.update(nodata=nodata)
os.makedirs("out", exist_ok=True)
with rasterio.open("out/mosaic_filled.tif", "w", **profile) as dst:
    dst.write(filled, 1)

Found 4 files to merge


In [4]:
from pathlib import Path
import rasterio

In [7]:
for tifs in Path("/Users/wm722/Documents/SCDs/data/Texas/Floyada_Texas").glob("*.tif"):
    with rasterio.open(tifs) as src:
        min_val = src.read(1).min()
        max_val = src.read(1).max()
        print(f"{tifs}: min={min_val}, max={max_val}")
    if False:
        if min_val < -1e8:
            # clip tif to -999999.0
            with rasterio.open(tifs) as src:
                data = src.read(1)
                data[data < -1e8] = -999999.0
                profile = src.profile
            with rasterio.open(tifs, "w", **profile) as dst:
                dst.write(data, 1)



/Users/wm722/Documents/SCDs/data/Texas/Floyada_Texas/USGS_one_meter_x30y379_TX_Panhandle_B8_2017.tif: min=-999999.0, max=967.02001953125
/Users/wm722/Documents/SCDs/data/Texas/Floyada_Texas/USGS_one_meter_x30y375_TX_Panhandle_B8_2017.tif: min=-999999.0, max=938.260009765625
/Users/wm722/Documents/SCDs/data/Texas/Floyada_Texas/USGS_one_meter_x27y377_TX_Panhandle_B8_2017.tif: min=963.8800048828125, max=992.4099731445312
/Users/wm722/Documents/SCDs/data/Texas/Floyada_Texas/USGS_one_meter_x29y377_TX_Panhandle_B8_2017.tif: min=944.260009765625, max=965.0800170898438
/Users/wm722/Documents/SCDs/data/Texas/Floyada_Texas/USGS_one_meter_x28y375_TX_Panhandle_B8_2017.tif: min=-999999.0, max=962.6699829101562
/Users/wm722/Documents/SCDs/data/Texas/Floyada_Texas/USGS_one_meter_x26y375_TX_Panhandle_B8_2017.tif: min=-999999.0, max=991.8599853515625
/Users/wm722/Documents/SCDs/data/Texas/Floyada_Texas/USGS_one_meter_x28y379_TX_Panhandle_B8_2017.tif: min=-999999.0, max=996.2999877929688
/Users/wm722/Do